In [ ]:
!pip install azure-ai-documentintelligence

In [ ]:
#part 1
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from google.colab import userdata

# 1. Configuration
ENDPOINT = "https://deepimgrec.cognitiveservices.azure.com/"
KEY = userdata.get('task1cp18april')  # Remember to rotate this!

def analyze_insurance_claim(file_path):
    # Initialize the client
    client = DocumentIntelligenceClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))

    # Read the file
    with open(file_path, "rb") as f:
        file_content = f.read()

    # 2. Analyze using the universal 'prebuilt-layout' model
    # The 'features' parameter is crucial for identifying form-like data
    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=AnalyzeDocumentRequest(bytes_source=file_content),
        features=["keyValuePairs"]
    )
    result = poller.result()

    # 3. Extraction Logic
    print("\n--- Extraction Results ---")

    if result.key_value_pairs:
        # Loop through found pairs to map them to your required variables
        for pair in result.key_value_pairs:
            key = pair.key.content if pair.key else "Unknown"
            value = pair.value.content if pair.value else "N/A"

            # Print all pairs found so you can identify your specific fields
            print(f"KEY: {key} | VALUE: {value}")
    else:
        print("No key-value pairs detected. The document structure may be too complex.")

if __name__ == "__main__":
    path = input("Enter the full path to your claim form: ").strip()
    if os.path.exists(path):
        analyze_insurance_claim(path)
    else:
        print("Error: File not found.")

Enter the full path to your claim form: /content/filledform2.png

--- Extraction Results ---
KEY: MEDICARE | VALUE: N/A
KEY: MEDICAID | VALUE: N/A
KEY: TRICARE
CHAMPUS (Sponsor's SSN) | VALUE: N/A
KEY: CHAMPVA | VALUE: N/A
KEY: GROUP
HEALTH PLAN | VALUE: N/A
KEY: FECA BLK LUNG | VALUE: N/A
KEY: OTHER | VALUE: N/A
KEY: INSURED'S I.D. NUMBER | VALUE: 987 65 4321A
KEY: (Medicare #) | VALUE: :selected:
KEY: (Medicaid #) | VALUE: :unselected:
KEY: (Member ID=)
Initial) | VALUE: :unselected:
KEY: (SSN ar ID) | VALUE: :unselected:
KEY: (SSM) | VALUE: :unselected:
KEY: (D) | VALUE: :unselected:
KEY: PATIENT'S NAME (Last Name, First Name, Middle | VALUE: DOE, JOHN S.
KEY: PATIENT'S BIRTH DATE | VALUE: 05 |01 |1925
KEY: INSURED'S NAME (Last Name, First Name, Middle Iniiali | VALUE: Leave Blank unless there is other
KEY: M | VALUE: :selected:
KEY: F | VALUE: :unselected:
KEY: PATIENT'S ADDRESS (No., Street) | VALUE: 10 Adams Street
KEY: INSURED'S ADDRESS Wo., Street) | VALUE: insurance primary to

In [ ]:
# part 2
import requests
import json
import os
from google.colab import userdata
search_key=userdata.get('task2')

search_service = "https://deepaisearch.search.windows.net"
index_name = "claims-index"
api_key = search_key

url = f"{search_service}/indexes/{index_name}/docs/index?api-version=2021-04-30-Preview"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# 🔥 Your extracted data
document = {
    "value": [
        {
            "@search.action": "upload",
            "id": "1",
            "policy_number": "POL123456",
            "name": "Rahul Sharma",
            "date": "12 March 2026",
            "claim_amount": 75000,
            "location": "Delhi",
            "previous_claims": "3"
        }
    ]
}

response = requests.post(url, headers=headers, data=json.dumps(document))

print(response.status_code)
print(response.json())

import requests

# 1. Configuration
search_url = "https://insaurance-claim-ankit.search.windows.net/indexes/claims-index/docs/search?api-version=2021-04-30-Preview"
search_key = 'C3a7FVOZeHOL0lZEK5bwV2QM6TktjjFOhyzXLHGxv0AzSeDMZa3z'

headers = {
    "Content-Type": "application/json",
    "api-key": search_key
}

def run_query(search_text, filter_expr=None):
    query = {"search": search_text}
    if filter_expr:
        query["filter"] = filter_expr

    response = requests.post(search_url, headers=headers, json=query)

    if response.status_code == 200:
        data = response.json()
        print(f"\n--- Results for '{search_text}' ---")
        for result in data.get("value", []):
            print(f"Name: {result.get('name')} | Amount: {result.get('claim_amount')} | Location: {result.get('location')}")
    else:
        print(f"Error {response.status_code}: {response.text}")

# 🎯 TASK A: Basic Search
run_query("Rahul Sharma")

# 🎯 TASK B: Search "Claims above 50,000"
run_query("*", filter_expr="claim_amount gt 50000")

# 🎯 TASK C: Search "Fraud cases in Delhi"
run_query("fraud", filter_expr="location eq 'Delhi'")


200
{'@odata.context': "https://deepaisearch.search.windows.net/indexes('claims-index')/$metadata#Collection(Microsoft.Azure.Search.V2021_04_30_Preview.IndexResult)", 'value': [{'key': '1', 'status': True, 'errorMessage': None, 'statusCode': 200}]}

--- Results for 'Rahul Sharma' ---
Name: Rahul Sharma | Amount: 75000 | Location: Delhi

--- Results for '*' ---
Name: Rahul Sharma | Amount: 75000 | Location: Delhi

--- Results for 'fraud' ---
